In [1]:
import pandas as pd,numpy as np,json
from sklearn.metrics.pairwise import cosine_distances
import getpass,os
from langchain.chat_models import init_chat_model
from ai_patterns_mining import parse_json_safe,Config
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from time import time
from langchain.tools import tool
from pydantic import BaseModel, Field
import seaborn as sns
import matplotlib.pyplot as plt
from langchain.agents.structured_output import ToolStrategy

/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)


In [3]:
llm.invoke("Hello, world!")

AIMessage(content='Hello there! How can I help you today?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'}, id='lc_run--e92f19b0-501e-4703-bed5-b19776a46141-0', usage_metadata={'input_tokens': 5, 'output_tokens': 42, 'total_tokens': 47, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 32}})

In [3]:
prompt = """\
You are an expert code description generator. Generate a concise and informative description of the community in 2-3 sentences. Focus on the code AI patterns. Strictly generate the description about patterns represented in the code. Avoid generic statements and ensure the description is specific to a AI patterns.

code:
{code}
"""

prompt2 = """\
You are an expert code description generator. Given a set of patterns that describe a community, generate a concise and informative description of the community in 2-3 sentences. Focus on the code patterns and code what does the community represent. Avoid generic statements and ensure the description is specific to the patterns provided.

code:
{code}
"""

In [4]:
def generate_community_description(code: str) -> str:
    response = llm.invoke(prompt.format(code=code))
    return response.content.strip()

In [5]:
community_result_json = "result/predicted_clusters_verification_results_with_descriptions.json"
community_results = json.load(open(community_result_json))
python_files = list(community_results.keys())

In [ ]:
def main(file):
    if community_results[file].get('code_summary') is not None:
        return 0
    code_snippets = open(f'result/repo_callgraph_clusters/{file}', 'r').read()
    description = generate_community_description(code_snippets)
    new_dict = community_results[file]
    new_dict['code_summary'] = description
    community_results[file] = new_dict
    json.dump(community_results, open("/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/notebooks/result/predicted_clusters_verification_results_with_descriptions.json", 'w'), indent=4)
    return 1

In [7]:
from concurrent.futures import ThreadPoolExecutor
start_time = time()
avg_time = 0
new_done = 0
done_count = 0
for file in python_files:
    total_time = time() - start_time
    avg_time = total_time / (new_done + 1)
    t = time()
    total_estimated = avg_time * (len(python_files) - done_count)
    hh,mm,ss = int(total_estimated)//3600, (int(total_estimated)%3600)//60, int(total_estimated)%60
    print(f" - Processing file: {file}, Estimated time remaining: {hh}h {mm}m {ss}s")
    done_count += 1
    new_done += main(file)
    
    

 - Processing file: 3DOD_thesis/cluster_0.py, Estimated time remaining: 0h 0m 1s
 - Processing file: 3DOD_thesis/cluster_1.py, Estimated time remaining: 0h 1m 50s
 - Processing file: 3DOD_thesis/cluster_10.py, Estimated time remaining: 0h 3m 32s
 - Processing file: 3DOD_thesis/cluster_2.py, Estimated time remaining: 0h 4m 34s
 - Processing file: 3DOD_thesis/cluster_3.py, Estimated time remaining: 0h 5m 34s
 - Processing file: 3DOD_thesis/cluster_4.py, Estimated time remaining: 0h 6m 40s
 - Processing file: 3DOD_thesis/cluster_5.py, Estimated time remaining: 0h 7m 46s
 - Processing file: 3DOD_thesis/cluster_6.py, Estimated time remaining: 0h 9m 19s
 - Processing file: 3DOD_thesis/cluster_7.py, Estimated time remaining: 0h 10m 24s
 - Processing file: 3DOD_thesis/cluster_8.py, Estimated time remaining: 0h 11m 18s
 - Processing file: 3DOD_thesis/cluster_9.py, Estimated time remaining: 0h 12m 26s
 - Processing file: AIlice/cluster_0.py, Estimated time remaining: 0h 13m 25s
 - Processing fil

In [36]:
json.dump(community_results, open("/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/notebooks/result/predicted_clusters_verification_results_with_descriptions.json", 'w'), indent=4)

In [19]:
done

['3DOD_thesis/cluster_0.py']

In [30]:
community_results['3DOD_thesis/cluster_0.py']

{'code_file': '3DOD_thesis/cluster_0.py',
 'predicted_label': 'Explainable AI (XAI) Techniques',
 'verification_result': {'is_correct': False,
  'score': 9,
  'explanation': 'The code snippet primarily focuses on data loading, preprocessing, and label generation for a 3D object detection/segmentation task. It involves geometric transformations and data preparation for a machine learning model. The provided pattern summaries describe techniques for explaining or interpreting *already trained* AI models (e.g., local explanations, feature importance, counterfactuals). The code does not implement any of these XAI techniques.'},
 'code_summary': 'This code implements a data pipeline for 3D object detection, specifically following the Frustum PointNet pattern. It processes LiDAR point clouds by projecting them into 2D image bounding box frustums, then transforms and normalizes these frustums into a canonical coordinate space. This pipeline generates comprehensive multi-task labels, including